**This is to show the data load from s3 and then create a bronze table in catalog**

In [0]:
%sql
-- Sanity check: see what Auto Loader is picking up
-- Does NOT write anything, just shows you the raw data
SELECT *
FROM datapipeline2026.engmarch2026.bronze_orders
LIMIT 10000;

In [0]:
%sql
-- Check what schema Auto Loader inferred from your JSON files
DESCRIBE  datapipeline2026.engmarch2026.bronze_orders;

In [0]:
# Cell 1 — Read JSON from table
bronze_df = spark.sql("SELECT * FROM datapipeline2026.engmarch2026.bronze_orders")

print("Row count:", bronze_df.count())
display(bronze_df)

In [0]:
# Cell 3 — Save bronze Delta table
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_orders")

print("Bronze table created")
spark.sql("SELECT COUNT(*) FROM bronze_orders").show()

In [0]:
# Cell 4 — Flatten to silver
from pyspark.sql import functions as F

silver_df = bronze_df.select(
    "order_id",
    F.col("customer.id").alias("customer__id"),
    F.col("customer.address.city").alias("customer__address__city"),
    F.col("customer.address.country_code").alias("customer__address__country_code"),
    F.col("customer.address.street").alias("customer__address__street"),
    F.col("customer.loyalty_tier").alias("customer__loyalty_tier"),
    F.col("customer.email").alias("customer__email"),
    F.col("payment.amount_cents").alias("payment__amount_cents"),
    F.col("payment.currency").alias("payment__currency"),
    F.explode_outer("items").alias("item")
).select(
    "order_id",
    "customer__id",
    "customer__address__city",
    "customer__address__country_code",
    "customer__address__street",
    "customer__loyalty_tier",
    "customer__email",
    "payment__amount_cents",
    "payment__currency",
    F.col("item.name").alias("item__name"),
    F.col("item.price").cast("double").alias("item__price"),
    F.col("item.qty").cast("int").alias("item__quantity"),
    F.col("item.sku").alias("item__sku"),
    F.current_timestamp().alias("_transformed_at")
)

display(silver_df)

In [0]:
# Cell 5 — Save silver Delta table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("datapipeline2026.engmarch2026.silver_orders")

print("Silver table created")
spark.sql("SELECT * FROM datapipeline2026.engmarch2026.silver_orders LIMIT 10").show()

In [0]:
%sql
SELECT * FROM datapipeline2026.engmarch2026.silver_orders LIMIT 10

In [0]:
from pyspark.sql import functions as F

silver_df = silver_df.withColumn(
    "_quality_score",
    (
        F.when(F.col("order_id").isNotNull(),        20).otherwise(0) +
        F.when(F.col("customer__id").isNotNull(),    20).otherwise(0) +
        F.when(F.col("customer__email").isNotNull(), 20).otherwise(0) +
        F.when(F.col("payment__amount_cents").isNotNull(), 20).otherwise(0) +
        F.when(F.col("item__sku").isNotNull(),20).otherwise(0)
    )
).withColumn(
    "_quality_grade",
    F.when(F.col("_quality_score") == 100, "A")
     .when(F.col("_quality_score") >= 80,  "B")
     .when(F.col("_quality_score") >= 60,  "C")
     .otherwise("F")
)

In [0]:
display(silver_df.limit(10))

In [0]:
silver_df = silver_df \
    .withColumn("payment__currency",
        F.upper(F.trim(F.col("payment__currency")))
    ) \
    .withColumn("customer__email_masked",
        F.regexp_replace(
            F.col("customer__email"),
            "(?<=.).(?=[^@]*@)", "*"   # masks email → j***@example.com
        )
    )

In [0]:
display(silver_df.limit(10))

In [0]:
silver_df = silver_df \
    .withColumn("item__total",
        F.round(F.col("item__quantity") * F.col("item__price"), 2)
    ) \
    .withColumn("item__discount_applied",
        F.when(F.col("item__price") < 50, "Y").otherwise("N")
    ) \
    .withColumn("order__value_tier",
        F.when(F.col("payment__amount_cents") >= 500, "high")
         .when(F.col("payment__amount_cents") >= 100, "medium")
         .otherwise("low")
    )

In [0]:
display(silver_df.limit(10))

In [0]:
silver_df = silver_df \
    .withColumn("created_at",
        F.to_timestamp(F.col("_transformed_at"))
    ) \
    .withColumn("order__year",
        F.year(F.col("_transformed_at"))
    ) \
    .withColumn("order__month",
        F.month(F.col("_transformed_at"))
    ) \
    .withColumn("order__day_of_week",
        F.date_format(F.col("_transformed_at"), "EEEE")  # Monday, Tuesday etc
    ) \
    .withColumn("order__hour",
        F.hour(F.col("_transformed_at"))
    ) \
    .withColumn("order__is_weekend",
        F.when(
            F.dayofweek(F.col("_transformed_at")).isin(1, 7), "Y"
        ).otherwise("N")
    )

In [0]:
display(silver_df.limit(10))

In [0]:
silver_df = silver_df \
    .fillna({
        "payment__currency": "USD",
        "item__quantity":    1,
        "item__price":       0.0
    }) \
    .withColumn("payment__amount",
        F.coalesce(F.col("payment__amount_cents"), F.lit(0.0))
    )

In [0]:
display(silver_df.limit(10))

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("datapipeline2026.engmarch2026.silver_mdf_orders")

print("Silver table saved")
spark.sql("SELECT COUNT(*) AS rows FROM datapipeline2026.engmarch2026.silver_mdf_orders").show()

In [0]:
%sql
use catalog datapipeline2026 ;

In [0]:
# Create or replace view for gold_country_orders using SQL
# Use spark.sql in Python cells for SQL operations
spark.sql("""
CREATE OR REPLACE VIEW datapipeline2026.engmarch2026.gold_country_orders AS
SELECT
    customer__address__country_code,
    COUNT(DISTINCT order_id)                    AS total_orders,
    SUM(item__total)                            AS total_revenue,
    ROUND(AVG(payment__amount), 2)              AS avg_order_value
FROM datapipeline2026.engmarch2026.silver_mdf_orders
GROUP BY customer__address__country_code
ORDER BY total_revenue DESC
""")